# Marketing Gold Aggregation
Run after Bronze and Silver in Databricks. This notebook builds six Delta snapshot tables for reporting, preserving the Silver table.

**Metric definitions**
- One accepted, unique campaign ID represents one campaign. Duplicate IDs stop the run rather than silently dropping records.
- CTR = total clicks / total impressions; zero impressions produces NULL. Rates are fractions: 0.05 means 5%.
- Average conversion rate and ROI are unweighted campaign averages, not overall conversion rate or portfolio ROI.
- Acquisition cost is excluded until its meaning and currency parsing are verified. No spend, revenue, or conversion counts are invented.
- Monthly summaries use the supplied campaign date. They are descriptive only until date provenance is verified; they do not allocate activity across the campaign duration.
- Rows failing Silver quality rules are excluded and counted. Passing those rules does not establish all business validity.

**Outputs:** `campaign_overview`, `channel_performance`, `campaign_type_performance`, `customer_segment_performance`, `monthly_channel_performance`, and `pipeline_quality_summary` in `marketing_gold` by default.

Each output is replaced on rerun. Writes are atomic per Delta table, not across all six tables. A common run ID identifies outputs from the same run. Run this notebook sequentially; avoid concurrent runs against the same targets.

In [ ]:
# 1. Configure table locations and acceptable quality loss.
from pyspark.sql import functions as F
from datetime import datetime, timezone
import re
import uuid

for name, default in {
    "catalog": "marketing_catalog",
    "silver_schema": "marketing_silver",
    "silver_table": "campaigns",
    "gold_schema": "marketing_gold",
    # Default: stop if any Silver row would be excluded.
    "max_rejected_fraction": "0.0",
}.items():
    dbutils.widgets.text(name, default)

catalog, silver_schema, silver_table, gold_schema = [
    dbutils.widgets.get(name).strip()
    for name in ["catalog", "silver_schema", "silver_table", "gold_schema"]
]
for value in [catalog, silver_schema, silver_table, gold_schema]:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value):
        raise ValueError(f"Invalid identifier: {value}")
if gold_schema == silver_schema:
    raise ValueError("Use separate Silver and Gold schemas")
max_rejected_fraction = float(dbutils.widgets.get("max_rejected_fraction"))
if not 0 <= max_rejected_fraction <= 1:
    raise ValueError("max_rejected_fraction must be between 0 and 1")
source_table = f"{catalog}.{silver_schema}.{silver_table}"
gold_prefix = f"{catalog}.{gold_schema}"
run_id = str(uuid.uuid4())
processed_at = datetime.now(timezone.utc)
print("Source:", source_table)
print("Gold schema:", gold_prefix)
print("Run ID:", run_id)


In [ ]:
# 2. Pin a Silver Delta version so all outputs read the same snapshot.
# VERSION AS OF avoids mixing data if Silver is overwritten during this run.
source_version = int(spark.sql(f"DESCRIBE HISTORY {source_table} LIMIT 1").first()["version"])
silver_df = spark.sql(f"SELECT * FROM {source_table} VERSION AS OF {source_version}")
required_types = {
    "campaign_id_typed": "bigint", "campaign_date": "date",
    "duration_days": "int", "conversion_rate_typed": "double",
    "roi_typed": "double", "clicks_typed": "bigint",
    "impressions_typed": "bigint", "engagement_score_typed": "int",
    "Channel_Used": "string", "Campaign_Type": "string",
    "Customer_Segment": "string", "_quality_passed": "boolean",
}
actual_types = dict(silver_df.dtypes)
errors = {name: {"expected": dtype, "actual": actual_types.get(name)}
          for name, dtype in required_types.items() if actual_types.get(name) != dtype}
if errors:
    raise ValueError(f"Silver schema mismatch: {errors}")
source_count = silver_df.count()
if source_count == 0:
    raise ValueError("Silver is empty; existing Gold tables will not be overwritten")
print("Silver version:", source_version)
print("Silver rows:", source_count)


In [ ]:
# 3. Apply Silver's quality flag, including NULL flags as rejected.
accepted_df = silver_df.filter(F.col("_quality_passed") == True)
accepted_count = accepted_df.count()
rejected_count = source_count - accepted_count
rejected_fraction = rejected_count / source_count
print(f"Accepted: {accepted_count}; rejected: {rejected_count} ({rejected_fraction:.2%})")
if accepted_count == 0 or rejected_fraction > max_rejected_fraction:
    raise ValueError("Quality gate failed. Investigate Silver before publishing Gold")

# Verify campaign grain and essential metric assumptions independently.
# This catches stale or incorrectly populated Silver quality flags.
invalid = (
    F.col("campaign_id_typed").isNull() | (F.col("campaign_id_typed") <= 0)
    | F.col("campaign_date").isNull()
    | F.col("duration_days").isNull() | (F.col("duration_days") <= 0)
    | F.col("clicks_typed").isNull() | (F.col("clicks_typed") < 0)
    | F.col("impressions_typed").isNull() | (F.col("impressions_typed") < 0)
    | (F.col("clicks_typed") > F.col("impressions_typed"))
    | F.col("engagement_score_typed").isNull()
)
for name in ["conversion_rate_typed", "roi_typed"]:
    invalid = invalid | F.col(name).isNull() | F.isnan(name) | (F.abs(F.col(name)) == float("inf"))
invalid = invalid | ~F.col("conversion_rate_typed").between(0, 1)
if accepted_df.filter(invalid).limit(1).count():
    raise ValueError("A quality-passed row violates Gold metric assumptions; repair Silver")
if accepted_df.groupBy("campaign_id_typed").count().filter(F.col("count") > 1).limit(1).count():
    raise ValueError("Repeated accepted campaign IDs: confirm campaign grain before aggregation")

# Trim grouping labels. NULL/blank categories remain NULL and form a visible group.
# Do not merge case variants without a verified business mapping.
def clean_label(name):
    value = F.trim(F.col(name))
    return F.when(F.length(value) > 0, value)

base_df = accepted_df.select(
    "campaign_id_typed", "campaign_date", "duration_days",
    "conversion_rate_typed", "roi_typed", "clicks_typed",
    "impressions_typed", "engagement_score_typed",
    clean_label("Channel_Used").alias("channel"),
    clean_label("Campaign_Type").alias("campaign_type"),
    clean_label("Customer_Segment").alias("customer_segment"),
).withColumn("campaign_month", F.trunc("campaign_date", "month"))
display(base_df.select(*[
    F.sum(F.when(F.col(name).isNull(), 1).otherwise(0)).alias(f"missing_{name}_rows")
    for name in ["channel", "campaign_type", "customer_segment"]
]))


In [ ]:
# 4. Define one consistent set of metrics for all reporting tables.
def aggregate_metrics(df, dimensions):
    return (df.groupBy(*dimensions).agg(
        F.count("*").alias("campaign_count"),
        F.sum("clicks_typed").alias("total_clicks"),
        F.sum("impressions_typed").alias("total_impressions"),
        F.avg("conversion_rate_typed").alias("avg_campaign_conversion_rate"),
        F.avg("roi_typed").alias("avg_campaign_roi"),
        F.avg("engagement_score_typed").alias("avg_engagement_score"),
        F.avg("duration_days").alias("avg_duration_days"),
        F.min("campaign_date").alias("earliest_campaign_date"),
        F.max("campaign_date").alias("latest_campaign_date"),
    ).withColumn("ctr", F.when(
        F.col("total_impressions") > 0,
        F.col("total_clicks").cast("double") / F.col("total_impressions")
    )))

# Keep stored precision; round only in dashboard presentation.
def add_audit(df):
    return (df.withColumn("_gold_run_id", F.lit(run_id))
        .withColumn("_gold_processed_at_utc", F.lit(processed_at).cast("timestamp"))
        .withColumn("_silver_source_table", F.lit(source_table))
        .withColumn("_silver_source_version", F.lit(source_version).cast("long")))

groupings = {
    "campaign_overview": [],
    "channel_performance": ["channel"],
    "campaign_type_performance": ["campaign_type"],
    "customer_segment_performance": ["customer_segment"],
    "monthly_channel_performance": ["campaign_month", "channel"],
}
outputs = {name: add_audit(aggregate_metrics(base_df, dimensions))
           for name, dimensions in groupings.items()}
quality_df = spark.createDataFrame(
    [(source_count, accepted_count, rejected_count, rejected_fraction, max_rejected_fraction)],
    "source_rows long, accepted_rows long, rejected_rows long, rejected_fraction double, max_rejected_fraction double",
)
outputs["pipeline_quality_summary"] = add_audit(quality_df)
for name in ["campaign_overview", "channel_performance", "pipeline_quality_summary"]:
    print(name)
    display(outputs[name])


In [ ]:
# 5. Validate every aggregation BEFORE writing any output.
# Each table is a separate breakdown of the same campaigns: never add totals
# from different Gold tables together, or campaigns will be counted repeatedly.
expected_totals = base_df.agg(
    F.count("*").alias("campaign_count"),
    F.sum("clicks_typed").alias("total_clicks"),
    F.sum("impressions_typed").alias("total_impressions"),
).first().asDict()

def validate_metrics(df, dimensions):
    totals = df.agg(*[F.sum(name).alias(name) for name in expected_totals]).first().asDict()
    if totals != expected_totals:
        raise ValueError(f"Aggregation totals mismatch: {totals}; expected {expected_totals}")
    if dimensions:
        if df.groupBy(*dimensions).count().filter(F.col("count") > 1).limit(1).count():
            raise ValueError(f"Duplicate Gold grouping keys: {dimensions}")
    elif df.count() != 1:
        raise ValueError("Overview must have exactly one row")
    invalid_ctr = (
        ((F.col("total_impressions") == 0) & F.col("ctr").isNotNull())
        | ((F.col("total_impressions") > 0) & (F.col("ctr").isNull() | ~F.col("ctr").between(0, 1)))
    )
    if df.filter(invalid_ctr).limit(1).count():
        raise ValueError("Invalid aggregate CTR")

for name, dimensions in groupings.items():
    validate_metrics(outputs[name], dimensions)
print("Pre-write validation: PASSED")


In [ ]:
# 6. Publish Delta snapshots, then verify the persisted data.
# Explicit select order and schema checks prevent accidental schema replacement.
# If a future schema change is intentional, migrate it explicitly before rerunning.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_prefix}")
for name, df in outputs.items():
    target = f"{gold_prefix}.{name}"
    if spark.catalog.tableExists(target):
        if spark.table(target).dtypes != df.dtypes:
            raise ValueError(f"Existing schema differs for {target}; review migration before writing")

for name, df in outputs.items():
    target = f"{gold_prefix}.{name}"
    df.write.format("delta").mode("overwrite").saveAsTable(target)
    saved = spark.table(target)
    if saved.dtypes != df.dtypes:
        raise RuntimeError(f"Saved schema mismatch: {target}")
    if saved.filter(F.col("_gold_run_id").isNull() | (F.col("_gold_run_id") != run_id)).limit(1).count():
        raise RuntimeError(f"Saved run ID mismatch: {target}")
    if name in groupings:
        validate_metrics(saved, groupings[name])
    elif saved.collect() != df.collect():
        raise RuntimeError("Saved quality summary mismatch")
    print(f"Verified {target}: {saved.count()} rows")
print("Gold publication and validation: PASSED")
print("Completed run:", run_id)


In [ ]:
# 7. Display reporting metrics with readable percentages.
# These rounded percentages are display-only; stored rates remain fractions.
channel_report = (spark.table(f"{gold_prefix}.channel_performance")
    .select(
        "channel", "campaign_count", "total_clicks", "total_impressions",
        F.round(F.col("ctr") * 100, 2).alias("ctr_percent"),
        F.round(F.col("avg_campaign_conversion_rate") * 100, 2).alias("avg_campaign_conversion_percent"),
        F.round("avg_campaign_roi", 3).alias("avg_campaign_roi"),
        F.round("avg_engagement_score", 2).alias("avg_engagement_score"),
        F.round("avg_duration_days", 2).alias("avg_duration_days"),
    ).orderBy(F.col("ctr_percent").desc_nulls_last(), "channel"))
display(channel_report)
display(spark.table(f"{gold_prefix}.monthly_channel_performance")
    .orderBy("campaign_month", "channel"))


## Using these tables
- Use `channel_performance` for a channel comparison dashboard and `monthly_channel_performance` for descriptive monthly trends.
- Display NULL categories as “Missing” in the dashboard without altering stored source labels.
- When rolling up monthly rows, recompute CTR from summed clicks and impressions. Weight campaign means by `campaign_count`; do not simply average group averages.
- Averages of conversion rates describe a typical campaign. An overall conversion rate requires verified conversion counts and denominators, which this dataset does not supply.
- Do not infer causal channel effectiveness from these descriptive summaries.
- Gold aggregates are reporting outputs. Campaign-level ML should use a separately designed feature dataset, a defined target, and leakage-aware splitting after date provenance is established.
- If publication fails midway, some tables may contain the new run and others an older run. Fix the error and rerun all cells; compare `_gold_run_id` across tables before refreshing a dashboard.